In [ ]:
import getpass

In [ ]:
ngrok_key= getpass.getpass('Enter ngrok key')

Enter ngrok key··········


In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle
from sklearn.preprocessing import StandardScaler

In [ ]:
df= pd.read_csv('claimants.csv')
df.head()

,CASENUM,ATTORNEY,CLMSEX,CLMINSUR,SEATBELT,CLMAGE,LOSS
0,5,0,0.0,1.0,0.0,50.0,34.940
1,3,1,1.0,0.0,0.0,18.0,0.891
2,66,1,0.0,1.0,0.0,5.0,0.330
3,70,0,0.0,1.0,1.0,31.0,0.037
4,96,1,0.0,1.0,0.0,30.0,0.038


In [ ]:
df.isnull().sum()

,0
CASENUM,0
ATTORNEY,0
CLMSEX,12
CLMINSUR,41
SEATBELT,48
CLMAGE,189
LOSS,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.fillna({'CLMSEX':df.CLMSEX.mode()[0],'CLMINSUR':df.CLMINSUR.mode()[0],'SEATBELT':df.SEATBELT.mode()[0],
          'CLMAGE':df.CLMAGE.median()},inplace=True)

In [ ]:
features= df.drop(columns=['CASENUM','ATTORNEY'])
features.head()

,CLMSEX,CLMINSUR,SEATBELT,CLMAGE,LOSS
0,0.0,1.0,0.0,50.0,34.940
1,1.0,0.0,0.0,18.0,0.891
2,0.0,1.0,0.0,5.0,0.330
3,0.0,1.0,1.0,31.0,0.037
4,0.0,1.0,0.0,30.0,0.038


In [ ]:
target= df[['ATTORNEY']]

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test= train_test_split(features,target,train_size=0.85,random_state=100,stratify=target)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(1139, 5)
(201, 5)
(1139, 1)
(201, 1)


In [ ]:
std_sca=StandardScaler()
x_train[['CLMAGE','LOSS']]= std_sca.fit_transform(x_train[['CLMAGE','LOSS']])
x_test[['CLMAGE','LOSS']]= std_sca.transform(x_test[['CLMAGE','LOSS']])

In [ ]:
log_model=LogisticRegression()
log_model.fit(x_train,y_train)
y_pred= log_model.predict(x_test)
accuracy_score(y_test,y_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.7114427860696517

In [ ]:
file='log.pkl'
pickle.dump(log_model,open(file,'wb'))

In [ ]:
scale= 'scale.pkl'
pickle.dump(std_sca,open(scale,'wb'))

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler
model=pickle.load(open('log.pkl','rb'))
scale= pickle.load(open('scale.pkl','rb'))
st.title('Insurance Claim Prediction')
def user_input_parameters():
    CLMSEX= st.sidebar.radio('Select your Gender',('Male','Female'))
    CLMINSUR= st.sidebar.selectbox('Enter details of Insurance, Yes-1,No-0',[0,1])
    SEATBELT= st.sidebar.selectbox('Enter details of Seatbelt, Yes-1, No-0',[0,1])
    CLMAGE= st.sidebar.slider('Select your age',0,100)
    LOSS= st.sidebar.number_input('Enter the loss')
    dict1= {'CLMSEX':CLMSEX,'CLMINSUR':CLMINSUR,'SEATBELT':SEATBELT,'CLMAGE':CLMAGE,'LOSS':LOSS}
    features= pd.DataFrame(dict1,index=[0])
    features['CLMSEX'] = features['CLMSEX'].map({'Male': 1, 'Female': 0})
    features[['CLMAGE','LOSS']]= scale.transform(features[['CLMAGE','LOSS']])
    return features
df= user_input_parameters()
pred= model.predict(df)
pred_prob= model.predict_proba(df)
button= st.button('predict')
if button is True:
    st.subheader('Prediction')
    st.write('Eligible' if pred_prob[0][1]>=0.5 else 'Not Eligible')
    st.subheader('Probabilities')
    st.write(pred_prob)

Overwriting app.py


In [ ]:
from pyngrok import ngrok
port =8501
ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://preobediently-unchanneled-juniper.ngrok-free.dev'

In [ ]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt